In [ ]:
from trafilatura.spider import focused_crawler

to_visit, known_urls = focused_crawler('https://www.vedomosti.ru', max_seen_urls=1)

In [ ]:
to_visit, len(to_visit)

In [ ]:
# from trafilatura import sitemaps

# urls = sitemaps.sitemap_search('https://www.vedomosti.ru')

In [ ]:
from trafilatura import sitemaps

urls_news = sitemaps.sitemap_search('https://www.vedomosti.ru/sitemap_news3.xml')

In [ ]:
# list([url for url in urls if '2025' in url])

In [ ]:
urls_news

In [ ]:
from dataclasses import dataclass
from datetime import datetime


@dataclass
class Article:
    url: str
    date: datetime
    topic: str
    title: str | None = None
    content: str | None = None
    tags: list[str] | None = None


In [ ]:
import re

vedomosti_url_date_pattern = re.compile(r'\/(\w+)\/news\/(\d{4})\/(\d{2})\/(\d{2})\/')

news_from_2022_04: list[Article] = []

for url in urls_news:
    match = vedomosti_url_date_pattern.search(url)
    if match:
        topic, year, month, day = match.groups()
        date = datetime(int(year), int(month), int(day))
        if date >= datetime(2022, 4, 1):
            news_from_2022_04.append(Article(url=url, date=date, topic=topic, tags=None, title=None, content=None))



In [ ]:
news_from_2022_04

In [ ]:
news_from_2022_04 = list(sorted(news_from_2022_04, key=lambda article: article.date))

In [ ]:
news_from_2022_04[0], news_from_2022_04[-1], len(news_from_2022_04)

In [ ]:
# Count articles per day
from collections import Counter

dates = [article.date for article in news_from_2022_04]
date_counts = Counter(dates)

# date_counts

In [ ]:
from matplotlib import pyplot as plt

# Plot the counts
plt.figure(figsize=(10, 6))
plt.bar(date_counts.keys(), date_counts.values(), width=0.5)
plt.xlabel('Date')
plt.ylabel('Number of Articles')
plt.title('Number of Articles per Day')
# Add total number of articles
plt.text(0.5, -0.1, f'Total number of articles: {len(news_from_2022_04)}', ha='center', va='top', transform=plt.gca().transAxes)
plt.show()

In [ ]:
# Parse articles using trafilatura

from trafilatura import fetch_url, extract
extract(fetch_url(news_from_2022_04[0].url))

In [ ]:
import urllib3
import logging
from concurrent.futures import ThreadPoolExecutor, as_completed
from trafilatura import fetch_url, extract

# Suppress the connection pool warnings - they're harmless
logging.getLogger('urllib3.connectionpool').setLevel(logging.ERROR)

def parse_article(i):
    if i % 100 == 0:
        print('Parsing article', i, 'of', len(news_from_2022_04))
    article = news_from_2022_04[i]
    article.content = extract(fetch_url(article.url))

with ThreadPoolExecutor(max_workers=30) as executor:
    futures = [executor.submit(parse_article, i) for i in range(len(news_from_2022_04))]
    for i, future in enumerate(as_completed(futures)):
        pass

news_from_2022_04[0].content

In [ ]:
# Save into csv file
import pandas as pd

df = pd.DataFrame([article.__dict__ for article in news_from_2022_04])
df.to_csv('news_from_2022_04.csv', index=False)



In [ ]:
from bs4 import BeautifulSoup

# Sitemap
with open('sitemap_news3.xml') as f:
    soup = BeautifulSoup(f.read(), "xml")

In [ ]:
urls = soup.find_all("url")

In [ ]:
from datetime import datetime

url_to_datetime_map = {}
for u in urls:
    loc = u.find("loc").text.strip()
    lastmod = u.find("lastmod").text.strip() if u.find("lastmod") else None

    # 4️⃣ Convert to datetime (ISO 8601)
    date = None
    if lastmod:
        try:
            date = datetime.fromisoformat(lastmod.replace("Z", "+00:00"))
        except ValueError:
            pass

    url_to_datetime_map[loc] = date

In [ ]:
url_to_datetime_map

In [ ]:
# Already parsed data
import pandas as pd
df = pd.read_csv('news_from_2022_04.csv')
print(df[:5])
df["date"] = df["url"].map(url_to_datetime_map).fillna(df["date"])
print(df[:5])

In [ ]:
# Remove title column
df = df.drop(columns=["title"])

In [ ]:
df.to_csv('news_vedomosti.csv', index=False)